# fix_grid

> Phase 13, Track A grids and the frozen analysis plan: `fixGrid` (the clue board: A1 pooling, A2 a liar), `profileGrid` (A3, the hidden-profile vote), their pilots and quick grids, `FIX_PLAN`, and the call and cost estimates (Phase 11's cached cells are free).

In [ ]:
#| default_exp fix_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy

from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueCell, clueLiarGrid, clueMainGrid
from explRL.llm_budget import PRICES, estimateCost
from explRL.llm_grid import HAIKU
from explRL.llm_runs import LLMConfig
from explRL.profile_runs import ARMS as PROFILE_ARMS
from explRL.profile_runs import profileCell

FIX_ARMS = dict(baseline={}, request=dict(ask=True), table=dict(table=True), evidence=dict(hint='evidence'),
                record=dict(record=True), verify=dict(check=True), warn=dict(hint='warn'))     # arm -> cfg.world keys
A1, A2 = ('baseline', 'request', 'table', 'evidence'), ('baseline', 'record', 'verify', 'warn')
A3, VARIANTS = tuple(PROFILE_ARMS), ('classic', 'lone')
KIND = dict(request='affordance', table='affordance', record='affordance', verify='affordance', factsFirst='affordance',
            evidence='instruction', warn='instruction', dissent='instruction')
FIX_SEEDS, PROFILE_SEEDS = range(900, 910), range(5000, 5100)
FIX_PLAN = dict(
    version=1, frozen=True, model=HAIKU, settings=CLUE_SETTINGS, prices=PRICES, turns=40,
    arms=dict(A1=list(A1), A2=list(A2), A3=list(A3), world=dict(FIX_ARMS), kind=dict(KIND),
              profile=dict(PROFILE_ARMS)),
    liar=dict(nLiar=1, slot=0, agent='ClueLiar (scripted, Phase 11)'),
    seeds=dict(fix=[900, 909], cached=[900, 904], cachedLiar=[900, 902], pilot=[900, 901], profile=[5000, 5099],
               profilePilot=[5000, 5019]),
    profile=dict(N=4, rounds=3, shared=[1, 1, 3], unique=2, variants=list(VARIANTS), pilotArms=['baseline', 'factsFirst'],
                 bands=dict(baseline=[0.2, 0.8], solo=0.8)),
    holm=dict(A1=[['request', 4], ['request', 8], ['table', 4], ['table', 8]], A2=['record', 'verify'],
              A3=[['factsFirst', 'classic'], ['factsFirst', 'lone'], ['table', 'classic'], ['table', 'lone']]),
    analyses=dict(
        A1='request and table: pooled-share gap closed vs baseline (mean paired difference on board seed / (1 - baseline '
           'mean)), N 4 and 8, bootstrap CI over seeds; Holm over the 4 tests (H13-1)',
        A1control='evidence (instruction) the same way',
        A2='record and verify: loss recovered = 1 - mean honest loss (welfareHonest clean - liar, same seed and arm) / '
           "baseline's, N 4, bootstrap CI over seeds; Holm over the 2 tests (H13-2)",
        A2control='warn the same way; each arm\'s no-liar welfare minus baseline\'s no-liar welfare, paired on seed',
        A3='factsFirst and table: win-rate gap closed vs baseline (mean paired difference on episode seed / (solo mean - '
           'baseline mean)), both variants, bootstrap CI over episodes; Holm over the 4 tests (H13-3)',
        A3control='dissent (instruction) the same way',
        mechanism='asks, checks, checked-false (turn logs); shares per held clue; private facts posted per private fact '
                  'held; votes against the hidden best'),
    nboot=2000, ci=0.95, exploratory=True)
FIX_ARMS.update(recordPrivate=dict(record='private'), verifyPublic=dict(check='public'))   # Phase 14 (FIX_PLAN keeps
KIND.update(recordPrivate='affordance', verifyPublic='affordance')                         # Phase 13's arms)


def _world(arm, s=CLUE_SETTINGS):
    return {**{k: s[k] for k in BOARD}, **FIX_ARMS[arm]}


def fixCell(study, arm, N, seed, liar=False, s=CLUE_SETTINGS, **cfg):
    """One clue-board cell of an arm (level `<arm>` or `<arm>|liar`); the baseline's world is exactly Phase 11's.
    liar: False, True (one liar) or the number of liars."""
    kw = dict(nLiar=int(liar)) if liar else {}
    return clueCell(study, arm + '|liar' * bool(liar), N, seed, s=s, channel='blackboard', world=_world(arm, s), **kw, **cfg)


def fixGrid(s=CLUE_SETTINGS, seeds=FIX_SEEDS):
    """Study fixA: A1 arms at N 4 and 8 and A2 arms at N 4 with one liar (slot 0) and without, 40 turns. The baseline's
    clean N 4 cells serve both parts (one cell each)."""
    cells = [fixCell('fixA', a, N, sd, s=s) for a in A1 for N in (4, 8) for sd in seeds]
    cells += [fixCell('fixA', a, 4, sd, liar, s) for a in A2 for liar in (False, True) for sd in seeds
              if liar or a != 'baseline']
    return cells


def fixPilotGrid(s=CLUE_SETTINGS):
    "Each new arm at N 4, seeds 900-901, 40 turns (the A2 arms with the liar): the main run replays these replies."
    return [fixCell('fixPilot', a, 4, sd, a in A2, s) for a in A1[1:] + A2[1:] for sd in (900, 901)]


def profileGrid(seeds=PROFILE_SEEDS, plan=FIX_PLAN, study='profileA', arms=A3, variants=VARIANTS):
    "Study profileA: arms x variants x episode seeds (shared by the arms), plus the solo ceiling on the same seeds."
    p = plan['profile']
    kw = dict(rounds=p['rounds'], N=p['N'], shared=list(p['shared']), unique=p['unique'])
    return [profileCell(study, a, v, sd, **kw) for a in (*arms, 'solo') for v in variants for sd in seeds]


def profilePilotGrid(plan=FIX_PLAN):
    "baseline and factsFirst, 20 episodes (classic), and the solo ceiling on the same 20."
    lo, hi = plan['seeds']['profilePilot']
    return profileGrid(range(lo, hi + 1), plan, 'profilePilot', tuple(plan['profile']['pilotArms']), ('classic',))


def fixQuick():
    "fixGrid's cells at seeds 900-901, 6 turns, for the fake-client pipeline check."
    return [dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=6)) for c in fixGrid(seeds=(900, 901))]


def profileQuick():
    "profileGrid's cells at four episode seeds."
    return profileGrid(range(5000, 5004))


HB_PLAN = dict(   # HiddenBench replication of A3 (specs/multiagent/phase13/hiddenbench.md)
    version=1, frozen=True, model=HAIKU, data='data/hiddenbench/benchmark.json', sha256='2815afffca4e470d', tasks=65,
    reps=3, seeds='episode seed = 1000 x task id + rep (shared by every arm and the solo ceiling)', N=4, rounds=3,
    arms=list(A3) + ['solo'], holm=['factsFirst', 'table'], control='dissent',
    bands=dict(invalid=0.05, solo=0.8, gap=0.2, pilotTasks=20),
    analyses=dict(primary='factsFirst and table: win-rate (strict majority) gap closed vs baseline, against the solo '
                          'ceiling, paired on episode seed, bootstrap over episodes; Holm over the 2 tests',
                  control='dissent (instruction) the same way',
                  secondary="agent accuracy (share of agents voting for the correct option, HiddenBench's default "
                            'metric) per arm; disclosure (hidden facts posted per hidden fact held)'),
    nboot=2000, ci=0.95, exploratory=True)


def hbGrid(plan=HB_PLAN, tasks=None, reps=None, arms=None, study='hbA'):
    "HiddenBench cells: arms (and the solo ceiling) x tasks x reps, one episode each, seeds shared across arms."
    ids = tasks or range(1, plan['tasks'] + 1)
    return [profileCell(study, a, 'hb', 1000 * i + r, rounds=plan['rounds'], N=plan['N'], bench='hiddenbench', item=i)
            for a in (arms or plan['arms']) for i in ids for r in range(plan['reps'] if reps is None else reps)]


def hbPilotGrid(plan=HB_PLAN):
    "The pilot: baseline and the solo ceiling on the first 20 tasks, one rep (the main run replays these replies)."
    return hbGrid(plan, range(1, plan['bands']['pilotTasks'] + 1), 1, ['baseline', 'solo'], 'hbPilot')


def phase11Cells():
    "Phase 11's blackboard cells the fix grid can replay (main N 4 and 8, seeds 900-904; liar, seeds 900-902)."
    return [c for c in clueMainGrid() + clueLiarGrid() if c['cfg']['channel'] == 'blackboard' and c['N'] in (4, 8)]


def requestKey(c):
    "What determines a cell's API requests: the config (N included) and the seed (not study, factor or level)."
    return (repr(sorted(c['cfg'].items())), c['seed'])


def cellCalls(c):
    "API calls of one cell: clue board (LLM agents x turns); vote (N x (rounds + 1)) or solo (1)."
    cfg = LLMConfig.fromDict(c['cfg'])
    if cfg.task == 'profile':
        return 1 if cfg.world.get('solo') else cfg.N * (cfg.turns + 1)
    return (cfg.N - cfg.nLiar) * cfg.turnsPerAgent()


def fixEstimate(cells, cached=None):
    "Cells, calls and expected cost, all and new (cells whose requests Phase 11 already made are cached, free)."
    cached = {requestKey(c) for c in (phase11Cells() if cached is None else cached)}
    out = dict(cells=len(cells), calls=0, newCalls=0, cost=0.0, newCost=0.0)
    for c in cells:
        n, cfg = cellCalls(c), c['cfg']
        cost = estimateCost(cfg['model'], cfg['task'], n, 1)
        new = requestKey(c) not in cached
        out['calls'] += n
        out['cost'] += cost
        out['newCalls'] += n * new
        out['newCost'] += cost * new
    return out

## Tests

In [ ]:
import json
from explRL.fix_grid import *
from explRL.bench_grid import rowKey
from explRL.clue_grid import BOARD, CLUE_SETTINGS, clueLiarGrid, clueMainGrid
from explRL.coop_grid import coopHash
from explRL.llm_runs import LLMConfig, cellRow

key = lambda c: rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0)))
g = fixGrid()
lv = lambda c: c['level']
a1 = [c for c in g if lv(c) in ('baseline', 'request', 'table', 'evidence')]
a2 = [c for c in g if lv(c) in ('record', 'verify', 'warn') or lv(c).endswith('|liar')]
# sizes: A1 4 arms x N {4, 8} x 10 seeds; A2 the other 3 arms clean and all 4 with the liar at N 4 (the baseline's clean
# N 4 cells are A1's), every cell 40 turns on the blackboard with Haiku and the Phase 11 board
assert len(g) == len(a1) + len(a2) == 80 + 70 and len({key(c) for c in g}) == len(g)
assert {c['seed'] for c in g} == set(range(900, 910)) and all(c['study'] == c['factor'] == 'fixA' for c in g)
for c in g:
    cfg, w = c['cfg'], c['cfg']['world']
    assert (cfg['channel'], cfg['turns'], cfg['model'], cfg['effort'], cfg['maxTokens']) == ('blackboard', 40, HAIKU, 'low', 8000)
    assert {k: w[k] for k in BOARD} == {k: CLUE_SETTINGS[k] for k in BOARD}
    assert {k: v for k, v in w.items() if k not in BOARD} == FIX_ARMS[lv(c).split('|')[0]]
    assert cfg['nLiar'] == lv(c).endswith('|liar') and (cfg['nLiar'] == 0 or c['N'] == 4)
assert FIX_ARMS['request'] == dict(ask=True) and FIX_ARMS['verify'] == dict(check=True) and FIX_ARMS['warn'] == dict(hint='warn')
# call counts and costs (design section 3): A1 about 17 000 new calls, A2 about 10 000
e1, e2 = fixEstimate(a1), fixEstimate(a2)
assert e1['calls'] == 19200 and e1['newCalls'] == 16800 and e2['calls'] == 9600 and e2['newCalls'] == 9600 - 360
assert 3.5 < e1['newCost'] < 4.5 and 1.8 < e2['newCost'] < 2.6

# The baseline cells are Phase 11's: everything that determines the API requests is equal (only study, factor and
# level differ), so the cached replies of seeds 900-904 (liar: 900-902) are reused
p11 = {(c['N'], c['seed'], c['cfg']['nLiar']): c for c in phase11Cells()}
base = [c for c in g if lv(c) in ('baseline', 'baseline|liar')]
same = [c for c in base if (c['N'], c['seed'], c['cfg']['nLiar']) in p11]
assert len(same) == 2 * 5 + 3 and all(c['seed'] <= 904 for c in same)
for c in same:
    ref = p11[(c['N'], c['seed'], c['cfg']['nLiar'])]
    assert c['cfg'] == ref['cfg'] and requestKey(c) == requestKey(ref) and (ref['study'], ref['level']) in (('clue', 'clue'), ('clueLiar', 'liar'))
    assert {k: v for k, v in c.items() if k not in ('study', 'factor', 'level')} == {k: v for k, v in ref.items() if k not in ('study', 'factor', 'level')}
assert {requestKey(c) for c in g if lv(c) not in ('baseline', 'baseline|liar')}.isdisjoint(map(requestKey, phase11Cells()))
# ... and they send the same requests (fake client, a few turns, clean and with the liar)
from explRL.clue_runs import runClue
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
def requests(c):
    cl = LLMClient(HAIKU, 'low', 8000, client=FakeClient(fakeReply))
    runClue(LLMConfig.fromDict(dict(c['cfg'], turns=3)), c['seed'], cl)
    return cl.client.requests
for c in (same[0], [c for c in same if c['cfg']['nLiar']][0]):
    ref = p11[(c['N'], c['seed'], c['cfg']['nLiar'])]
    assert requests(c) == requests(ref) and len(requests(c)) == 3 * (c['N'] - c['cfg']['nLiar'])
# the new arms do change the requests (schema with 'ask' / 'check', rules, observations)
assert requests(fixCell('x', 'request', 4, 900)) != requests(same[0])

# pilot: each new arm at N 4, seeds 900-901, A2 arms with the liar; the main grid replays its cells
p = fixPilotGrid()
assert sorted({lv(c) for c in p}) == sorted(['request', 'table', 'evidence', 'record|liar', 'verify|liar', 'warn|liar'])
assert len(p) == 12 and {c['seed'] for c in p} == {900, 901} and {c['N'] for c in p} == {4}
assert {requestKey(c) for c in p} <= {requestKey(c) for c in g} and all(c['study'] == 'fixPilot' for c in p)

# the vote: 4 arms x 2 variants x 100 episodes plus the solo ceiling, episode seeds shared by every arm
v = profileGrid()
assert len(v) == (4 + 1) * 2 * 100 and {c['level'] for c in v} == {f'{a}|{x}' for a in (*A3, 'solo') for x in VARIANTS}
for lev in {c['level'] for c in v}:
    assert sorted(c['seed'] for c in v if c['level'] == lev) == list(range(5000, 5100))
w = v[0]['cfg']['world']
assert (w['shared'], w['unique'], v[0]['cfg']['turns'], v[0]['cfg']['N']) == ([1, 1, 3], 2, 3, 4) and all(c['study'] == 'profileA' for c in v)
ev = fixEstimate(v)
assert ev['calls'] == 4 * 2 * 100 * 16 + 2 * 100 == 13000 and ev['newCalls'] == 13000 and 2 < ev['cost'] < 4
pp = profilePilotGrid()
assert len(pp) == 3 * 20 and {c['level'] for c in pp} == {'baseline|classic', 'factsFirst|classic', 'solo|classic'}
assert {c['seed'] for c in pp} == set(range(5000, 5020)) and {requestKey(c) for c in pp} <= {requestKey(c) for c in v}
# pilots: about 3 000 calls in all, under $1
ep = [fixEstimate(p), fixEstimate(pp)]
assert sum(x['calls'] for x in ep) == 1680 + 660 and sum(x['cost'] for x in ep) < 1
# a plan change (the pilot's fact counts) reaches the grid
alt = dict(FIX_PLAN, profile=dict(FIX_PLAN['profile'], shared=[1, 1, 2], unique=3))
assert profilePilotGrid(alt)[0]['cfg']['world']['shared'] == [1, 1, 2] and profilePilotGrid(alt)[0]['cfg']['world']['unique'] == 3

# quick grids: every fix cell kind at two seeds and 6 turns; every vote level at four episodes
q, pq = fixQuick(), profileQuick()
assert len(q) == 30 and all(c['cfg']['turns'] == 6 for c in q) and {lv(c) for c in q} == {lv(c) for c in g}
assert len(pq) == 40 and {c['level'] for c in pq} == {c['level'] for c in v} and all(c['cfg']['turns'] == 40 for c in g)
# FIX_PLAN: version 1, not frozen yet, JSON-serialisable with a stable hash
assert FIX_PLAN['version'] == 1 and FIX_PLAN['frozen'] is True and coopHash(FIX_PLAN) == '79abfb531cdd'
# the HiddenBench replication's grids (AC-3b): 65 tasks x 3 reps x (4 arms + solo); seeds shared across arms; the pilot
hb, hp = hbGrid(), hbPilotGrid()
assert len(hb) == 65 * 3 * 5 and len(hp) == 40 and HB_PLAN['frozen'] is True and coopHash(HB_PLAN) == '791a2df079a8'
seeds = lambda arm: sorted(c['seed'] for c in hb if c['level'] == f'{arm}|hb')
assert all(seeds(a) == seeds('baseline') for a in HB_PLAN['arms']) and len(set(seeds('baseline'))) == 195
assert all(c['cfg']['world']['bench'] == 'hiddenbench' and c['cfg']['world']['item'] == c['seed'] // 1000 for c in hb)
assert {c['level'] for c in hp} == {'baseline|hb', 'solo|hb'} and {c['seed'] // 1000 for c in hp} == set(range(1, 21))
assert coopHash(json.loads(json.dumps(FIX_PLAN))) == coopHash(FIX_PLAN)
assert FIX_PLAN['holm']['A1'] == [['request', 4], ['request', 8], ['table', 4], ['table', 8]] and FIX_PLAN['holm']['A2'] == ['record', 'verify']
assert set(FIX_PLAN['analyses']) == {'A1', 'A1control', 'A2', 'A2control', 'A3', 'A3control', 'mechanism'}
# Phase 14: two more arms (FIX_PLAN keeps Phase 13's arms and hash); fixCell takes a number of liars
assert FIX_ARMS['recordPrivate'] == dict(record='private') and FIX_ARMS['verifyPublic'] == dict(check='public')
assert KIND['recordPrivate'] == KIND['verifyPublic'] == 'affordance' and len(FIX_ARMS) == 9
assert 'recordPrivate' not in FIX_PLAN['arms']['world'] and 'verifyPublic' not in FIX_PLAN['arms']['kind']
c2 = fixCell('x', 'record', 8, 900, 2)
assert c2['cfg']['nLiar'] == 2 and c2['level'] == 'record|liar' and fixCell('x', 'record', 4, 900, True) == fixCell('x', 'record', 4, 900, 1)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()